# SSL, Fine Tuning, and Linear Probing Heads

> Good luck!

In [ ]:
#| default_exp heads

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F

from torch import nn
import math
from physiojepa.utils import trunc_normal_
from torch.nn.attention import SDPBackend, sdpa_kernel
from physiojepa.jepa import MLP, JEPABlock

In [ ]:
#| export
class CrossAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=12,
        qkv_bias=False,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, q, x):
        q = self.W_Q(q).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        k = self.W_K(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH], set_priority=True):
            q = F.scaled_dot_product_attention(q, k, v)
        q = q.transpose(1, 2).flatten(-2)
        q = self.proj(q)
        return q


class CrossAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.,
        qkv_bias=False,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.xattn = CrossAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

    def forward(self, q, x):
        y = self.xattn(q, self.norm1(x))
        q = q + y
        q = q + self.mlp(self.norm2(q))
        return q

class AttentivePooler(nn.Module):
    """ Attentive Pooler """
    def __init__(
        self,
        num_queries=1,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        complete_block=True,
    ):
        super().__init__()
        self.query_tokens = nn.Parameter(torch.zeros(1, num_queries, embed_dim))

        self.complete_block = complete_block
        if complete_block:
            self.cross_attention_block = CrossAttentionBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                norm_layer=norm_layer)
        else:
            self.cross_attention_block = CrossAttention(
                dim=embed_dim,
                num_heads=num_heads,
                qkv_bias=qkv_bias)

        self.blocks = None
        if depth > 1:
            self.blocks = nn.ModuleList([
                JEPABlock(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    qk_scale=False,
                    norm_layer=norm_layer)
                for i in range(depth-1)])

        self.init_std = init_std
        trunc_normal_(self.query_tokens, std=self.init_std)
        self.apply(self._init_weights)
        self._rescale_blocks()

    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        if self.complete_block:
            rescale(self.cross_attention_block.xattn.proj.weight.data, 1)
            rescale(self.cross_attention_block.mlp.fc2.weight.data, 1)
        else:
            rescale(self.cross_attention_block.proj.weight.data, 1)
        if self.blocks is not None:
            for layer_id, layer in enumerate(self.blocks, 1):
                rescale(layer.attn.proj.weight.data, layer_id + 1)
                rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        q = self.query_tokens.repeat(len(x), 1, 1)
        q = self.cross_attention_block(q, x)
        if self.blocks is not None:
            for blk in self.blocks:
                q = blk(q)
        return q

class AttentiveClassifier(nn.Module):
    """ Attentive Classifier """
    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        num_classes=1000,
        complete_block=True,
        num_queries=1,
        affine=False,
        c_in=7,
    ):
        super().__init__()
        self.affine = affine
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(c_in,1,1))
            self.affine_bias = nn.Parameter(torch.zeros(c_in,1,1))
        self.pooler = AttentivePooler(
            num_queries=num_queries,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            depth=depth,
            norm_layer=norm_layer,
            init_std=init_std,
            qkv_bias=qkv_bias,
            complete_block=complete_block,
        )

        
        self.linear = nn.Linear(embed_dim*c_in, num_classes, bias=True)

    def forward(self, x):
        """
        x: [bs x nvars x d_model x num_patch] 
        out: [bs x n_classes]
        """
        x = x.transpose(2,3) # [bs x nvars x num_patch x d_model]
        B, nvars, n_patches, d_model = x.shape
        if self.affine:
            x = x * self.affine_weight + self.affine_bias
        x = torch.reshape(x, (B * nvars, n_patches, d_model)) # u: [bs * nvars x num_patch x d_model]
        x = self.pooler(x)#.squeeze(1)
       
        x = torch.reshape(x, (-1, nvars, d_model)) # z: [bs x nvars x num_patch x d_model]
        x = x.flatten(start_dim=-2)
        x = self.linear(x)
        if x.dim() == 3:
            x = x.transpose(1, 2)
        return x

In [ ]:
#| export
class AttentiveClassifierNoMelt(nn.Module):
    """ Attentive Classifier """
    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        num_classes=1000,
        complete_block=True,
        num_queries=1,
        affine=False,
        c_in=7,
    ):
        super().__init__()
        self.pooler = AttentivePooler(
            num_queries=num_queries,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            depth=depth,
            norm_layer=norm_layer,
            init_std=init_std,
            qkv_bias=qkv_bias,
            complete_block=complete_block,
        )

        
        self.linear = nn.Linear(embed_dim, num_classes, bias=True)

    def forward(self, x):
        """
        x: [bs x nvars * num_patch x d_model] 
        out: [bs x n_classes]
        """
        x = self.pooler(x)#.squeeze(1)
        x = x.flatten(start_dim=-2)
        x = self.linear(x)
        return x

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()